# 01 — Cleaning

Transform raw data files in `../data/raw/` into a clean SQLite database
at `../data/clean/housing.db`.

**Inputs:**
- Redfin: `nj_mercer_4zips.tsv` — quarterly market tracker for 4 ZIPs
- Zillow: `zillow_zhvi_4zips.csv` — monthly home value index (wide format)
- Zillow: `zillow_zori_4zips.csv` — monthly rent index (wide format)
- Census: `census_acs5_2022_4zips.json` — demographics
- Census: `census_acs5_2022_4zips_taxes.json` — property taxes

**Outputs (in `housing.db`):**
- `redfin_monthly` — Redfin time series
- `zhvi_monthly` — Zillow home values, long format
- `zori_monthly` — Zillow rents, long format
- `demographics` — Census demographics (one row per ZIP)
- `taxes` — Census taxes (one row per ZIP)

**Target ZIPs:** 08648 Lawrenceville, 08534 Pennington, 08619 Hamilton, 08550 Princeton Jct

In [ ]:
import pandas as pd
import sqlite3
import json
from pathlib import Path

# Paths — one place to change if the project ever moves
PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / "data" / "raw"
CLEAN = PROJECT_ROOT / "data" / "clean"
DB_PATH = CLEAN / "housing.db"

# Sanity check — confirm we can see the data
print(f"Raw data folder: {RAW}")
print(f"Clean data folder: {CLEAN}")
print(f"Database target:  {DB_PATH}")
print()
print("Files in raw/:")
for f in sorted(RAW.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name} ({size_mb:.1f} MB)")

In [2]:
# Load Redfin filtered TSV into a pandas DataFrame
redfin_raw = pd.read_csv(RAW / "nj_mercer_4zips.tsv", sep="\t")

print(f"Shape: {redfin_raw.shape[0]:,} rows × {redfin_raw.shape[1]} columns")
print(f"First 8 columns: {list(redfin_raw.columns[:8])}")
print()
redfin_raw.head(3)

Shape: 2,794 rows × 58 columns
First 8 columns: ['PERIOD_BEGIN', 'PERIOD_END', 'PERIOD_DURATION', 'REGION_TYPE', 'REGION_TYPE_ID', 'TABLE_ID', 'IS_SEASONALLY_ADJUSTED', 'REGION']



,PERIOD_BEGIN,PERIOD_END,PERIOD_DURATION,REGION_TYPE,REGION_TYPE_ID,TABLE_ID,IS_SEASONALLY_ADJUSTED,REGION,CITY,STATE,...,SOLD_ABOVE_LIST_YOY,PRICE_DROPS,PRICE_DROPS_MOM,PRICE_DROPS_YOY,OFF_MARKET_IN_TWO_WEEKS,OFF_MARKET_IN_TWO_WEEKS_MOM,OFF_MARKET_IN_TWO_WEEKS_YOY,PARENT_METRO_REGION,PARENT_METRO_REGION_METRO_CODE,LAST_UPDATED
0,2025-07-01,2025-09-30,90,zip code,2,3130,False,Zip Code: 08648,NaN,New Jersey,...,-0.087302,NaN,NaN,NaN,0.372881,-0.062603,-0.076098,"Trenton, NJ",45940,2026-04-14 14:25:48.665 Z
1,2021-10-01,2021-12-31,90,zip code,2,3087,False,Zip Code: 08534,NaN,New Jersey,...,0.234921,NaN,NaN,NaN,0.410256,-0.079940,0.247466,"Trenton, NJ",45940,2026-04-14 14:25:48.665 Z
2,2025-05-01,2025-07-31,90,zip code,2,3095,False,Zip Code: 08550,NaN,New Jersey,...,-1.000000,NaN,NaN,NaN,0.285714,-0.168831,-0.214286,"Trenton, NJ",45940,2026-04-14 14:25:48.665 Z


In [4]:
# Clean Redfin: filter to "All Residential", extract zip, parse dates,
# keep only the columns we need, rename to lowercase
redfin = (
    redfin_raw
    .query('PROPERTY_TYPE == "All Residential"')
    .assign(
        zip=lambda df: df['REGION'].str.replace('Zip Code: ', '', regex=False),
        period_begin=lambda df: pd.to_datetime(df['PERIOD_BEGIN']),
        period_end=lambda df: pd.to_datetime(df['PERIOD_END']),
    )
    [['zip', 'period_begin', 'period_end', 'PERIOD_DURATION',
      'MEDIAN_SALE_PRICE', 'MEDIAN_PPSF', 'HOMES_SOLD',
      'NEW_LISTINGS', 'INVENTORY', 'MEDIAN_DOM', 'MONTHS_OF_SUPPLY',
      'PENDING_SALES', 'MEDIAN_LIST_PRICE']]
    .rename(columns=str.lower)
    .rename(columns={'period_duration': 'period_days'})
    .sort_values(['zip', 'period_begin'])
    .reset_index(drop=True)
)

print(f"Cleaned shape: {redfin.shape[0]:,} rows × {redfin.shape[1]} columns")
print(f"ZIPs:          {sorted(redfin['zip'].unique())}")
print(f"Date range:    {redfin['period_begin'].min().date()} to {redfin['period_begin'].max().date()}")
print()
redfin.head(3)

Cleaned shape: 676 rows × 13 columns
ZIPs:          ['08534', '08550', '08619', '08648']
Date range:    2012-01-01 to 2026-01-01



,zip,period_begin,period_end,period_days,median_sale_price,median_ppsf,homes_sold,new_listings,inventory,median_dom,months_of_supply,pending_sales,median_list_price
0,08534,2012-01-01,2012-03-31,90,322500.0,170.760935,26,74.0,107.0,190.0,NaN,29.0,397450.0
1,08534,2012-02-01,2012-04-30,90,310500.0,170.760935,22,102.0,124.0,248.0,NaN,31.0,450000.0
2,08534,2012-03-01,2012-05-31,90,297500.0,163.775510,28,102.0,115.0,167.0,NaN,45.0,456000.0


In [ ]:
# Zillow arrives in WIDE format: one ROW per ZIP, one COLUMN per month.
# We need LONG format: one ROW per (zip, month) pair.
# The pandas method for this transformation is called `melt`.

# --- ZHVI: Zillow Home Value Index ---
zhvi_wide = pd.read_csv(RAW / "zillow_zhvi_4zips.csv", dtype={'RegionName': str})
month_cols_zhvi = [c for c in zhvi_wide.columns if c[:4].isdigit() and '-' in c]

zhvi = (
    zhvi_wide
    .melt(id_vars=['RegionName'], value_vars=month_cols_zhvi,
          var_name='month', value_name='zhvi')
    .rename(columns={'RegionName': 'zip'})
    .assign(month=lambda df: pd.to_datetime(df['month']))
    .dropna(subset=['zhvi'])
    .sort_values(['zip', 'month'])
    .reset_index(drop=True)
)

# --- ZORI: Zillow Observed Rent Index (same pattern) ---
zori_wide = pd.read_csv(RAW / "zillow_zori_4zips.csv", dtype={'RegionName': str})
month_cols_zori = [c for c in zori_wide.columns if c[:4].isdigit() and '-' in c]

zori = (
    zori_wide
    .melt(id_vars=['RegionName'], value_vars=month_cols_zori,
          var_name='month', value_name='zori')
    .rename(columns={'RegionName': 'zip'})
    .assign(month=lambda df: pd.to_datetime(df['month']))
    .dropna(subset=['zori'])
    .sort_values(['zip', 'month'])
    .reset_index(drop=True)
)

print(f"ZHVI: {zhvi.shape[0]:,} rows, {zhvi['zip'].nunique()} zips, "
      f"{zhvi['month'].min().date()} → {zhvi['month'].max().date()}")
print(f"ZORI: {zori.shape[0]:,} rows, {zori['zip'].nunique()} zips, "
      f"{zori['month'].min().date()} → {zori['month'].max().date()}")
print()
print("Latest ZHVI per zip:")
zhvi.groupby('zip').last()[['month', 'zhvi']]

In [ ]:
# Census returns JSON as a 2D list: first row is column names, rest are data.
# Load it, make a DataFrame, rename Census codes to human names, coerce types.
with open(RAW / "census_acs5_2022_4zips.json") as f:
    rows = json.load(f)
demographics_raw = pd.DataFrame(rows[1:], columns=rows[0])

demographics = (
    demographics_raw
    .rename(columns={
        'zip code tabulation area': 'zip',
        'B01003_001E': 'population',
        'B19013_001E': 'median_hh_income',
        'B01002_001E': 'median_age',
        'B25003_001E': 'total_housing_units',
        'B25003_002E': 'owner_occupied_units',
        'B25077_001E': 'median_home_value',
        'B25064_001E': 'median_gross_rent',
        'B15003_022E': 'bachelors_degree_holders',
    })
    .drop(columns=['NAME'])
    .astype({
        'population': int, 'median_hh_income': int, 'median_age': float,
        'total_housing_units': int, 'owner_occupied_units': int,
        'median_home_value': int, 'median_gross_rent': int,
        'bachelors_degree_holders': int,
    })
    .assign(pct_owner_occupied=lambda df: (df['owner_occupied_units'] / df['total_housing_units']).round(3))
    .sort_values('zip')
    .reset_index(drop=True)
)

print(f"Demographics shape: {demographics.shape}")
demographics

In [ ]:
# Census taxes — same pattern, different codes.
# B25103_001E = median real estate taxes paid (annual dollars)
# B25105_001E = median monthly selected owner costs
# B25077_001E = median home value (for computing effective rate)
with open(RAW / "census_acs5_2022_4zips_taxes.json") as f:
    rows = json.load(f)
taxes_raw = pd.DataFrame(rows[1:], columns=rows[0])

taxes = (
    taxes_raw
    .rename(columns={
        'zip code tabulation area': 'zip',
        'B25103_001E': 'median_real_estate_taxes',
        'B25105_001E': 'median_monthly_owner_costs',
        'B25077_001E': 'median_home_value',
    })
    .drop(columns=['NAME'])
    .astype({
        'median_real_estate_taxes': int,
        'median_monthly_owner_costs': int,
        'median_home_value': int,
    })
    .assign(
        effective_tax_rate_pct=lambda df:
            (df['median_real_estate_taxes'] / df['median_home_value'] * 100).round(2)
    )
    .sort_values('zip')
    .reset_index(drop=True)
)

# Note: Census top-codes taxes at $10,001 for privacy — so 08534 and 08550
# show "10001" but are actually higher. Flag in a column so we don't forget.
taxes['taxes_topcoded'] = taxes['median_real_estate_taxes'] >= 10001

print(f"Taxes shape: {taxes.shape}")
taxes

In [ ]:
# Write all 5 DataFrames into a SQLite database.
# The database becomes our single source of truth from here on.
CLEAN.mkdir(parents=True, exist_ok=True)

# Delete any existing DB file so every run produces a fresh reproducible copy
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)

# .to_sql() writes a DataFrame as a table. if_exists='replace' drops+recreates.
# index=False means "don't write the pandas row index as a column".
redfin.to_sql('redfin_monthly',  conn, if_exists='replace', index=False)
zhvi.to_sql  ('zhvi_monthly',    conn, if_exists='replace', index=False)
zori.to_sql  ('zori_monthly',    conn, if_exists='replace', index=False)
demographics.to_sql('demographics', conn, if_exists='replace', index=False)
taxes.to_sql ('taxes',           conn, if_exists='replace', index=False)

# Verify — query sqlite's internal catalog to list the tables we just made
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name",
    conn
)
print("Tables in housing.db:")
print(tables.to_string(index=False))

# Row count per table
print("\nRow counts:")
for t in tables['name']:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", conn)['n'][0]
    print(f"  {t:<18} {n:>6,} rows")

conn.close()
print(f"\nDatabase written: {DB_PATH}")
print(f"Size: {DB_PATH.stat().st_size / 1024:.1f} KB")

In [ ]:
# Victory lap — prove the DB works by joining 4 tables into a single summary.
# This is roughly the answer to Q4 (neighborhood tax/price comparison), one query.
conn = sqlite3.connect(DB_PATH)

query = """
WITH latest_redfin AS (
    SELECT zip, median_sale_price, median_ppsf, homes_sold
    FROM redfin_monthly r1
    WHERE period_begin = (
        SELECT MAX(period_begin) FROM redfin_monthly r2 WHERE r2.zip = r1.zip
    )
),
latest_zhvi AS (
    SELECT zip, zhvi
    FROM zhvi_monthly z1
    WHERE month = (
        SELECT MAX(month) FROM zhvi_monthly z2 WHERE z2.zip = z1.zip
    )
)
SELECT
    d.zip,
    d.population,
    d.median_hh_income AS income,
    d.pct_owner_occupied AS owner_pct,
    lr.median_sale_price AS redfin_latest_sale,
    lz.zhvi AS zhvi_latest,
    t.median_real_estate_taxes AS annual_taxes,
    t.effective_tax_rate_pct AS tax_rate_pct,
    t.taxes_topcoded
FROM demographics d
LEFT JOIN latest_redfin lr ON d.zip = lr.zip
LEFT JOIN latest_zhvi   lz ON d.zip = lz.zip
LEFT JOIN taxes         t  ON d.zip = t.zip
ORDER BY d.zip
"""

result = pd.read_sql(query, conn)
conn.close()

# Drop in readable names so the output reads like a spreadsheet a human would build
zip_names = {
    '08534': 'Pennington',
    '08550': 'Princeton Jct',
    '08619': 'Hamilton',
    '08648': 'Lawrenceville',
}
result.insert(1, 'name', result['zip'].map(zip_names))

print("Neighborhood summary — 4 tables joined in one SQL query:")
result